<img src='../OUTILS/bandeau_MF.png' align='right' width='100%'/>


# <div style='background-color: #27ae60; color: white; padding: 20px; border-radius: 10px; text-align: center;'>🌍🛰️ Élaboration d'un produit RGB à partir de NetCDF – MTG / FCI)</div>

---

Exploration du NetCDF, Extraction de canaux, Fabrication d'un produit RGB

---

<div class="alert alert-block alert-warning">
    
⚠️ <b>PRÉREQUIS</b>
    
Aucun prérequis technique spécifique. La librairie GDAL et ImageMagick sont préinstallées dans l'environnement `env_MF_teledetection`.

</div>
<hr>

<div class="alert alert-info" role="alert">
<h3> ⚙️ Initialisation de l'environnement</h3>
Importation des librairies et configuration des chemins d'accès.
</div>

In [ ]:
from PIL import Image
from IPython.display import display, HTML
import time
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import os
import subprocess
import sys
from osgeo import gdal

os.environ['PATH'] = f"/opt/conda/env_MF_teledetection/bin:{os.environ['PATH']}" 
os.environ['PATH'] = f"~/.conda/envs/env_MF_teledetection/bin:{os.environ['PATH']}"
os.environ['GDAL_DATA'] = '/opt/conda/env_MF_teledetection/share/gdal'
os.environ['PROJ_LIB'] = '/opt/conda/env_MF_teledetection/share/proj'

## Préparation des répertoires

Définition des chemins d'entrée/sortie et création du dossier `RESULTS`.

In [ ]:
cd ~/MF_DATA_MANIPULATION

In [ ]:
download_dir = os.path.join(os.getcwd(), "RESULTS")
os.makedirs(download_dir, exist_ok=True)

#❗ Attention : ce chemin doit-être modifié pour toute arborescence différente
input = '../stockage/DATA/NetCDF/Mmultic2kmNC4_mtgi1_202605181200.nc'
#input = '/stockage/DATA/NetCDF/Mmultic2kmNC4_mtgi1_202605181200.nc'
output = 'RESULTS'

print(f"📁 Données source : {input}")
print(f"📁 Dossier de sortie : {output}")

In [ ]:
!pwd

<div class="alert alert-info alert-success">
<h3> 1️⃣ - 🔎 Analyse du fichier NetCDF</h3>
</div>

La commande `gdalinfo` donne un aperçu des métadonnées, des sous-datasets (bandes) et des projections.

In [ ]:
!gdalinfo {input} 2>/dev/null

In [ ]:
!gdalinfo -mm NETCDF:"{input}":VIS006 | grep -E "Size|Computed Min/Max|Pixel Size" 2>/dev/null

<div class="alert alert-info alert-success">
<h3> 2️⃣ - 🖼️ Extraction d’un canal simple (VIS0.64)</h3>
</div>

On extrait une bande du NetCDF vers un GeoTIFF, puis on crée une vignette JPG pour visualisation.

In [ ]:
!gdal_translate -q -ot byte -scale 0 10000 0 255 NETCDF:"$input":VIS006 {output}/VIS0.64.tif 2>/dev/null
!gdalwarp -q -overwrite -ts 500 500 {output}/VIS0.64.tif {output}/VIS0.64.jpg 2>/dev/null
im0 = Image.open(output + '/VIS0.64.jpg')
display(im0)

<div class="alert alert-info alert-success">
<h3> 3️⃣ - 🎨 Fabrication du RGB « Cloud Type »</h3>
</div>

La recette du produit Cloud Type RGB est la suivante :

<img src='../OUTILS/cloudtype_tableau.png' align='left' width='40%'/>

<div class="alert alert-info" role="alert">


La recette officielle (EUMETSAT) associe :

| Canal | Rôle | Plage dynamique | Gamma |
|-------|------|----------------|-------|
| **NIR1.38** (IR_013) | Rouge | 0 % → 10 % | 1.5 |
| **VIS0.64** (VIS006) | Vert | 0 % → 80 % | 0.75 |
| **NIR1.61** (IR_016) | Bleu | 0 % → 80 % | 1.0 |


</div>

### <a href="../DOCS/RGB-WS-2025_MeetingReport_final.pdf" target="_blank">Lien vers les recettes RGB du Workshop</a> 

### 🔴 Étape A - Bande rouge à partir du canal du NIR1.38

Extraction et étirement </br>
Plage 0 → 10 % (valeur brute 1000).</br>
Gamma = 1.5 </br>
Option : -ot byte : spécifie le type de données de sortie en 8 bits. Ici limitation à une plage de 0 à 255 </br>

In [ ]:
gamma=1.5
ValGamma=1/gamma
!gdal_translate -q -ot byte -scale 0 1000 0 255 -exponent {ValGamma} NETCDF:"$input":IR_013 {output}/NIR1.38_scale_gamma.tif
!gdalwarp -q -overwrite -ts 1000 1000 {output}/NIR1.38_scale_gamma.tif {output}/NIR1.38_scale_gamma.jpg
im2 = Image.open(output + '/NIR1.38_scale_gamma.jpg')
display(im2)

💡 Remarque : </br>
**Gdal_calc** permet également d'appliquer le gamma grâce à la formule du gamma</br>
Voici un exemple de commande :</br>
**Récupérer le min et le max** de valeur de l'image :
file_in = 'RESULTS/NIR1.38_scale.tif'</br>
imgpil = Image.open(file_in)</br>
img = np.array(imgpil)</br>
max_val = np.max(img)</br>
min_val = np.min(img)</br>
**Appliquer le calcul**</br>
Formule : `out = max_in * ((in - min_in) / (max_in - min_in))^(1/gamma)` </br>
!gdal_calc.py --quiet --NoDataValue 0 --overwrite -A {file_in} --calc="numpy.maximum(A*0.,{max_val}*((A-{min_val})/({max_val}-{min_val}))**(1/{gamma}))" --outfile {output}/NIR1.38_scale_gamma.tif

### 🟢 Étape B - Bande verte à partir du canal VIS 0.64 
Plage 0 → 80 % (valeur brute 8000).</br>
Gamma = 0.75

In [ ]:
gamma=0.75
ValGamma=1/gamma
!gdal_translate -ot byte -scale 0 8000 0 255 -exponent {ValGamma} NETCDF:"$input":VIS006 {output}/VIS0.64_scale_gamma.tif 2>/dev/null
!gdalwarp -overwrite -ts 1000 1000 {output}/VIS0.64_scale_gamma.tif {output}/VIS0.64_scale_gamma.jpg
im3 = Image.open(output + '/VIS0.64_scale_gamma.jpg')
display(im3)

### 🔵 Étape C - Bande bleue à partir du canal NIR 1.61
Plage 0 → 80 % (valeur brute 8000).</br>
Gamma = 1 donc pas de correction

In [ ]:
!gdal_translate -ot byte -scale 0 8000 0 255 NETCDF:"$input":IR_016 {output}/NIR1.61_scale_gamma.tif 2>/dev/null
!gdalwarp -overwrite -ts 1000 1000 {output}/NIR1.61_scale_gamma.tif {output}/NIR1.61_scale_gamma.jpg
im4 = Image.open(output + '/NIR1.61_scale_gamma.jpg')
display(im4)

<div class="alert alert-info" role="alert">
    
## 🔴 Étape D - Fusion des couches (R, G, B)

`gdal_merge.py -separate` assemble trois fichiers en une image RVB.

</div>

In [ ]:
!gdal_merge.py -separate  {output}/NIR1.38_scale_gamma.tif {output}/VIS0.64_scale_gamma.tif  {output}/NIR1.61_scale_gamma.tif -o {output}/RGB_cloudtype.tif
!convert -resize 1000 {output}/RGB_cloudtype.tif {output}/RGB_cloudtype_min.jpg > /dev/null 2>&1
im5 = Image.open(output + '/RGB_cloudtype_min.jpg')
display(im5)

⚠️ La recette complète pour ce produit indique qu'il est nécessaire d'appliquer une correction de l'angle solaire à chaque canaux. (Récupération du signal pour les zones avec angle solaire faible)

<div class="alert alert-info alert-success">
<h3> 4️⃣ - 🗺️ Découpes et reprojections »</h3>
</div>


On peut extraire une sous‑zone (par exemple la région Afrique de l’Ouest) et la reprojeter dans différents systèmes.



### 🌍 Découpe en WGS84 (EPSG:4326)

In [ ]:
nord = 40
sud = 0
ouest = -23
est = 8

In [ ]:
!gdalwarp -q -t_srs "EPSG:4326" -te {ouest} {sud} {est} {nord} -overwrite {output}/RGB_cloudtype.tif {output}/RGB_cloudtype_zoom.tif
!convert -resize 500 {output}/RGB_cloudtype_zoom.tif {output}/RGB_cloudtype_zoom_min.jpg > /dev/null 2>&1
im6 = Image.open(output + '/RGB_cloudtype_zoom_min.jpg')
display(im6)

### 🗺️ Projection orthographique (vue satellite centrée sur la France)

In [ ]:
latitude = 45
longitude = 1
taille_domaine = "-1720000 -990000 1720000 990000"

In [ ]:
!gdalwarp -q -overwrite -t_srs "+proj=ortho lat_0={latitude} lon_0={longitude}" -te {taille_domaine} {output}/RGB_cloudtype.tif {output}/RGB_cloudtype_zoom_ortho.tif
!gdal_rasterize -b 1 -burn 250 -b 2 -burn 250 -b 3 -burn 250 -l world-administrative-boundaries OUTILS/boundary/world-administrative-boundaries.shp {output}/RGB_cloudtype_zoom_ortho.tif >/dev/null 2>&1
!convert -resize 800 {output}/RGB_cloudtype_zoom_ortho.tif {output}/RGB_cloudtype_zoom_ortho_min.jpg >/dev/null 2>&1
im7 = Image.open(output + '/RGB_cloudtype_zoom_ortho_min.jpg')
display(im7)

### 🧭 Projection Mercator (EPSG:3857)

Les coordonnées doivent être converties en **mètres**. On utilise `cs2cs`.

In [ ]:
!echo {ouest} {sud} | cs2cs +init=epsg:4326 +to +init=epsg:3857
!echo {est} {nord} | cs2cs +init=epsg:4326 +to +init=epsg:3857

In [ ]:
!gdalwarp -t_srs "EPSG:3857" -te -2560348.29 0 890555.93 4865942.28 -overwrite {output}/RGB_cloudtype_zoom.tif {output}/RGB_cloudtype_zoom_3857.tif
!convert -resize 500 {output}/RGB_cloudtype_zoom_3857.tif {output}/RGB_cloudtype_zoom_3857_min.jpg >/dev/null 2>&1
im9 = Image.open(output + '/RGB_cloudtype_zoom_3857_min.jpg')
display(im9)

<div class="alert alert-info alert-success">
<h3> 5️⃣ - 🔎  Zoom sur une perturbation</h3>
</div>


In [ ]:
!gdalwarp -q -t_srs "EPSG:4326" -te -40 30 10 65 -overwrite {output}/RGB_cloudtype.tif {output}/RGB_cloudtype_zoom_perturb.tif
!convert -resize 1200 {output}/RGB_cloudtype_zoom_perturb.tif {output}/RGB_cloudtype_zoom_perturb.jpg >/dev/null 2>&1
im8 = Image.open(output + '/RGB_cloudtype_zoom_perturb.jpg')
display(im8)

### <div style='background-color: #872459; color: white; padding: 10px; border-radius: 12px; text-align: left;'> ✏️ Exercice</div>

### Produire le produit Cloud Phase RGB et faire une découpe 

La recette du produit Cloud Phase RGB est la suivante :</br>
<img src='../OUTILS/tableau_cloud_phase.jpg' align='left' width='40%'/>

<div class="alert alert-info" role="alert">


La recette officielle Cloud Phase RGB (EUMETSAT) associe :

| Canal | Rôle | Plage dynamique | Gamma |
|-------|------|----------------|-------|
| **NIR1.61** | Rouge | 0 % → 50 % | 1.0 |
| **NIR2.25**  | Vert | 0 % → 50 % | 1.0 |
| **VIS0.64** | Bleu | 0 % → 100 % | 1.0 |


</div>

## 🧹 Nettoyage des fichiers intermédiaires (optionnel)

La cellule ci-dessous supprime tous les `.tif` et `.xml` temporaires pour libérer de l’espace.

In [ ]:
!rm {output}/*tif {output}/*xml 2>/dev/null

---

✅ **Fin du TP** – Vous savez maintenant créer un RGB à partir d’un NetCDF MTG, appliquer des corrections gamma, et reprojeter/découper des images satellites.

📚 **Ressources** : [Documentation GDAL](https://gdal.org) 
